# <center> MODELADO V2 (Ensembles)

In [1]:
import joblib
from sklearn.pipeline import Pipeline
import pandas as pd
from xgboost import XGBRegressor, plot_importance
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna
import numpy as np
from sklearn.model_selection import GridSearchCV


# Cargar datasets procesados
train = pd.read_csv('../data/processed/train.csv')
test = pd.read_csv('../data/processed/test.csv')

# Separar X e y
X_train = train.drop(columns='SalePrice')
y_train = train['SalePrice']

X_test = test.drop(columns='SalePrice')
y_test = test['SalePrice']

c:\Users\User\OneDrive\Escritorio\RoadMap-DataScientist\AmesHousing-Models\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Entrenamiento XGBRegressor
xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

In [3]:
# Entrenamiento LGBMRegressor
lgbm = LGBMRegressor(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42)
lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000106 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 781
[LightGBM] [Info] Number of data points in the train set: 2051, number of used features: 5
[LightGBM] [Info] Start training from score 12.011152
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [4]:
# Entrenamiento CatBoostRegressor
cat = CatBoostRegressor(iterations=200, learning_rate=0.1, depth=6, random_state=42, verbose=0)
cat.fit(X_train, y_train)
y_pred_cat = cat.predict(X_test)

In [5]:
# Evaluación de modelos
def evaluar_modelo(nombre, y_true, y_pred):
    print(f"📌 {nombre}")
    print(f"MAE: {mean_absolute_error(y_true, y_pred):.3f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.3f}")
    print(f"R2: {r2_score(y_true, y_pred):.3f}")
    print("-"*30)

evaluar_modelo("XGBoost", y_test, y_pred_xgb)
evaluar_modelo("LightGBM", y_test, y_pred_lgbm)
evaluar_modelo("CatBoost", y_test, y_pred_cat)

📌 XGBoost
MAE: 0.115
RMSE: 0.161
R2: 0.851
------------------------------
📌 LightGBM
MAE: 0.112
RMSE: 0.158
R2: 0.856
------------------------------
📌 CatBoost
MAE: 0.109
RMSE: 0.150
R2: 0.870
------------------------------


In [6]:
# GridSearchCV para XGBoost
param_grid_xgb = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0, 0.5],
    'reg_lambda': [1, 2]
}

grid_xgb = GridSearchCV(estimator=xgb, param_grid=param_grid_xgb,
                        scoring='neg_mean_squared_error',
                        cv=3, n_jobs=-1, verbose=1)

grid_xgb.fit(X_train, y_train)
print("Best XGB params:", grid_xgb.best_params_)


Fitting 3 folds for each of 432 candidates, totalling 1296 fits
Best XGB params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'reg_alpha': 0.5, 'reg_lambda': 2, 'subsample': 1.0}


In [7]:
# GridSearchCV para LightGBM

param_grid_lgbm = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'num_leaves': [20, 50, 100],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_lgbm = GridSearchCV(estimator=lgbm, param_grid=param_grid_lgbm,
                         scoring='neg_mean_squared_error',
                         cv=3, n_jobs=-1, verbose=1)

grid_lgbm.fit(X_train, y_train)
print("Best LGBM params:", grid_lgbm.best_params_)

Fitting 3 folds for each of 324 candidates, totalling 972 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000106 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 781
[LightGBM] [Info] Number of data points in the train set: 2051, number of used features: 5
[LightGBM] [Info] Start training from score 12.011152
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

In [8]:
# GridSearchCV para CatBoost

param_grid_cat = {
    'iterations': [200, 500],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5]
}

grid_cat = GridSearchCV(estimator=cat, param_grid=param_grid_cat,
                        scoring='neg_mean_squared_error',
                        cv=3, n_jobs=-1, verbose=1)

grid_cat.fit(X_train, y_train)
print("Best CatBoost params:", grid_cat.best_params_)

Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best CatBoost params: {'depth': 6, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.01}


In [9]:
models = {
    'XGB': grid_xgb.best_estimator_,
    'LGBM': grid_lgbm.best_estimator_,
    'CatBoost': grid_cat.best_estimator_
}

for name, model in models.items():
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    print(f'{name} MSE: {mse:.4f}')

XGB MSE: 0.0232
LGBM MSE: 0.0242
CatBoost MSE: 0.0231
